In [18]:
import pandas as pd
import numpy as np
import sys
import os
from tqdm import tqdm
import sys

In [19]:
sys.path.append('../src/')
sys.path.append('../src/scripts_shibani')

In [20]:
from time_series import *

In [4]:
from utils import load_sessions, read_session
from events import generate_event_seq
from summary import stats, print_summary_stats
from main import generate_buffer

In [5]:
import warnings 
warnings.filterwarnings("ignore")

# Compute writer-GenAI collaboration metrics per time window

In [6]:
#Load data
sessions = load_sessions()

Successfully downloaded 1447 writing sessions in CoAuthor!


In [7]:
err = []
file_name = []
text = []
sentence_metrics_list = []
api_metrics_list = []

In [8]:
all_time_window_metrics = []

for sess in tqdm(sessions):
    events = read_session(sess, verbose=0)
    if events is None:
        continue

    events = pd.DataFrame(events)

    # Compute metrics for each time window
    if events is None or len(events) == 0:
        continue

    events = pd.DataFrame(events)
    if events.empty:
        continue

    try:
        time_window_metrics = compute_metrics_per_time_window(events)
    except Exception as exc:
        err.append((sess, repr(exc)))
        continue

    if time_window_metrics is None or time_window_metrics.empty:
        continue

    # Add session ID to the time window metrics
    session_id = sess.split("/")[-1].rstrip(".jsonl")
    time_window_metrics["session_id"] = session_id

    # Append to the global list
    all_time_window_metrics.append(time_window_metrics)

100%|██████████| 1447/1447 [02:11<00:00, 10.98it/s]


In [9]:
# Combine all time window metrics into a single DataFrame
time_window_metrics_df = pd.concat(all_time_window_metrics, ignore_index=True)

In [12]:
# Filter writing sessions with only one time window
timeframes_count = pd.DataFrame(time_window_metrics_df['session_id'].value_counts())
single_window_sessions = timeframes_count[timeframes_count['count'] == 1].index.to_list()
time_window_metrics_df = time_window_metrics_df[~time_window_metrics_df['session_id'].isin(single_window_sessions)]

In [23]:
# Set the "session_id" as the first column of the dataframe
time_window_metrics_df = time_window_metrics_df[
    ["session_id"] + [col for col in time_window_metrics_df.columns if col != "session_id"]
]

In [24]:
# Save time windows data
time_window_metrics_df.to_csv('../data/temporal_windows_metrics.csv', index=None)